<a href="https://colab.research.google.com/github/Shatha-1/Data-Science/blob/main/Phase2_cleaned_openFDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df = pd.read_json("openfda_influenza_raw.json")

In [2]:
# 2) Quick inspection
print("Raw shape:", df.shape)
print("Duplicates (safetyreportid):", df["safetyreportid"].duplicated().sum())
print("\nTop missing columns:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

Raw shape: (1000, 27)
Duplicates (safetyreportid): 0

Top missing columns:
seriousnesscongenitalanomali    976
seriousnessdisabling            974
authoritynumb                   968
seriousnesslifethreatening      961
seriousnessdeath                935
seriousnesshospitalization      651
seriousnessother                502
occurcountry                     40
primarysourcecountry             24
duplicate                        11
dtype: int64


In [3]:
# 3) Flatten patient
patient_df = pd.json_normalize(df["patient"])
patient_df.columns = [f"patient_{c}" for c in patient_df.columns]
df = pd.concat([df.drop(columns=["patient"]), patient_df], axis=1)

print("\nAfter patient flatten:", df.shape)


After patient flatten: (1000, 34)


In [4]:
# 4) Explode reactions (because it's a list)
df_exp = df.explode("patient_reaction").reset_index(drop=True)
print("After reaction explode:", df_exp.shape)

After reaction explode: (7566, 34)


In [5]:
# 5) Flatten reaction dict -> get reaction name
reaction_df = pd.json_normalize(df_exp["patient_reaction"])
reaction_df.columns = [f"reaction_{c}" for c in reaction_df.columns]
df_exp = pd.concat([df_exp.drop(columns=["patient_reaction"]), reaction_df], axis=1)

In [6]:
# 6) Clean reaction text
df_exp["reaction_reactionmeddrapt"] = (
    df_exp["reaction_reactionmeddrapt"]
    .astype(str)
    .str.lower()
    .str.strip()
)
df_exp.loc[df_exp["reaction_reactionmeddrapt"].isin(["nan", "none", ""]), "reaction_reactionmeddrapt"] = pd.NA

In [7]:
# 7) Convert dates
df_exp["receivedate"] = pd.to_datetime(df_exp["receivedate"].astype(str), errors="coerce")
df_exp["transmissiondate"] = pd.to_datetime(df_exp["transmissiondate"].astype(str), errors="coerce")

In [8]:
# 8) Clean age (numeric + valid range)
df_exp["patient_patientonsetage"] = pd.to_numeric(df_exp["patient_patientonsetage"], errors="coerce")
df_exp.loc[(df_exp["patient_patientonsetage"] < 0) | (df_exp["patient_patientonsetage"] > 120), "patient_patientonsetage"] = pd.NA


In [9]:
# 9) Keep only useful columns (structured dataset)
keep_cols = [
    "safetyreportid", "primarysourcecountry", "occurcountry",
    "serious", "receivedate", "transmissiondate",
    "patient_patientonsetage", "patient_patientonsetageunit",
    "patient_patientsex", "patient_patientweight",
    "reaction_reactionmeddrapt"
]
df_clean = df_exp[keep_cols].copy()

In [11]:
# drop rows with no reaction name (usually needed for reaction frequency)
df_clean = df_clean.dropna(subset=["reaction_reactionmeddrapt"])

print("\nFinal cleaned shape:", df_clean.shape)
print("Missing in key columns:")
print(df_clean[["reaction_reactionmeddrapt","receivedate","patient_patientonsetage"]].isnull().sum())

# 10) Save cleaned structured file
df_clean.to_csv("openfda_influenza_clean.csv", index=False)
print("\nSaved: openfda_influenza_clean.csv")


Final cleaned shape: (7566, 11)
Missing in key columns:
reaction_reactionmeddrapt       0
receivedate                     0
patient_patientonsetage      1394
dtype: int64

Saved: openfda_influenza_clean.csv
